# 머신러닝 기반 텍스트 분류
1. 데이터 준비
2. 학습-평가
3. 배포 준비

In [13]:
import pandas as pd
datafile = 'data/Korean_movie_reviews_2016.csv'
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [14]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [15]:
review_list = list(data_df.review)
label_list = list(data_df.label)

In [16]:
from sklearn.model_selection import train_test_split

train_X, text_X, train_Y, test_Y = train_test_split(review_list, label_list, test_size=0.1)
len(train_X), len(text_X), len(train_Y), len(test_Y)

(148845, 16539, 148845, 16539)

In [17]:
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = set(['Noun', 'Verb', 'Adjective'])
    my_stopwords = set('하는 한다 의하여 하여 있다 하며 하여야'.split())
    return [word for word, tag in Okt().pos(text) if tag in my_tags and word not in my_stopwords]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
vectorizer.fit(train_X) # 학습, 단어사전 만들기
# print(vectorizer.get_feature_names_out())
# # 특징 벡터 추출
# sample_dtm = vectorizer.transform(sample_corpus) # 문제풀기
# sample_dtm.toarray()

TfidfVectorizer(max_features=1000)

In [75]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가고', '가는', '가면', '가볍', '가서', '가슴', '가장', '가족', '가지', '가치'],
       dtype=object))

In [ ]:
train_X_fv = vectorizer.transform(train_X) # 모델에 집어넣기 위해서 숫자 행렬로 변환

In [77]:
text_X_fv = vectorizer.transform(text_X)

In [78]:
train_X_fv

<148845x1000 sparse matrix of type '<class 'numpy.float64'>'
	with 817779 stored elements in Compressed Sparse Row format>

In [ ]:
import numpy as np
train_Y = np.array(train_Y) # 학습효율 증대
tesy_Y = np.array(test_Y)
train_Y[:10]

array([1, 1, 0, 1, 0, 1, 1, 1, 1, 0])

# 2.머신러닝 - 모델 학습
1. 의사결정트리, Decision Tree(DT)
2. 랜덤포레스트, RandomForest(RF)

In [80]:
# 모델별 정확도를 dataframe으로 저장해서 비교
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [ ]:
def get_scores(model, train_X, train_Y, test_X, test_Y):
    train_score = model.score(train_X, train_Y) * 100
    test_score = model.score(test_X, test_Y) * 100
    return train_score, test_score

## 1. Decision Tree

In [83]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_Y)

DecisionTreeClassifier()

In [28]:
train_score, test_score = get_scores(dtc, train_X_fv, train_Y, text_X_fv, test_Y)
print(train_score, test_score)

98.10406799019114 79.80530866436906


In [29]:
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.104068,79.805309


## 2. Random Forest

In [81]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, bootstrap=True, n_jobs=-1) # n_jobs=-1 : cpu풀로 다돌리기
rf.fit(train_X_fv, train_Y)

RandomForestClassifier(n_jobs=-1)

In [82]:
train_score, test_score = get_scores(rf, train_X_fv, train_Y, text_X_fv, test_Y)
print(train_score, test_score)
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

95.81712519735295 83.86843219057984


,train,test
RandomForest,95.817125,83.868432


## 3. 나이브 베이즈

In [32]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(train_X_fv, train_Y)

MultinomialNB()

In [33]:
train_score, test_score = get_scores(mnb, train_X_fv, train_Y, text_X_fv, test_Y)
print(train_score, test_score)
score_df.loc['NaiveBayes'] = [train_score, test_score]
score_df

85.3572508314018 84.63026785174435


,train,test
DecisionTree,98.104068,79.805309
RandomForest,98.102724,85.059556
NaiveBayes,85.357251,84.630268


## 4. 로지스틱 회귀

In [34]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_Y)

LogisticRegression(solver='liblinear')

In [35]:
train_score, test_score = get_scores(lr, train_X_fv, train_Y, text_X_fv, test_Y)
print(train_score, test_score)
score_df.loc['LinearRegression'] = [train_score, test_score]
score_df

86.23131445463402 85.34373299473971


,train,test
DecisionTree,98.104068,79.805309
RandomForest,98.102724,85.059556
NaiveBayes,85.357251,84.630268
LinearRegression,86.231314,85.343733


## 5. SVM

In [ ]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(train_X_fv, train_Y)

LinearSVC()

In [39]:
train_score, test_score = get_scores(svm, train_X_fv, train_Y, text_X_fv, test_Y)
print(train_score, test_score)
score_df.loc['SVM'] = [train_score, test_score]
score_df

86.22258053680002 85.21071406977447


,train,test
DecisionTree,98.104068,79.805309
RandomForest,98.102724,85.059556
NaiveBayes,85.357251,84.630268
LinearRegression,86.231314,85.343733
SVM,86.222581,85.210714


In [42]:
score_df.sort_values(by='test', ascending=False)

,train,test
LinearRegression,86.231314,85.343733
SVM,86.222581,85.210714
RandomForest,98.102724,85.059556
NaiveBayes,85.357251,84.630268
DecisionTree,98.104068,79.805309


## 6. 배포준비
- 기능 구현
- 모델 저장

In [84]:
from konlpy.tag import Okt
review = '이게 영화냐? 나도 만들겠다'

def analyze_sentiment(review):
    my_tags = set(['Noun', 'Verb', 'Adjective'])
    my_stopwords = set('하는 한다 의하여 하여 있다 하며 하여야'.split())
    token_review = ' '.join(word for word, tag in Okt().pos(review) if tag in my_tags and word not in my_stopwords)
    # 전처리 및 특징 벡터 추출
    review_fv = vectorizer.transform([token_review])
    # print(review_fv)

    result = rf.predict(review_fv)
    # print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'
    return show

In [85]:
show = analyze_sentiment(review)
print(f'{review} -> {show}')

이게 영화냐? 나도 만들겠다 -> 부정


In [86]:
reviews = {
    '영화가 너무 재미있다',
    '이게 영화냐? 나도 만들겠다',
    '개노잼',
    '개꿀잼'
}
for review in reviews:
    print(f'{review} -> {analyze_sentiment(review)}')

영화가 너무 재미있다 -> 긍정
개노잼 -> 부정
개꿀잼 -> 긍정
이게 영화냐? 나도 만들겠다 -> 부정


In [87]:
import joblib

vectorizer_file = 'model/sa_movie_vectorizer.pkl'
model_file = 'model/sa_movie_model.pkl'
# joblib.load -> 역직렬화. 파일을 파이썬 객체로 만듦
# joblib.dump -> 직렬화. 객체를 파일형식으로 만듦
joblib.dump(vectorizer, vectorizer_file)
joblib.dump(rf, model_file)

['model/sa_movie_model.pkl']